# 🛰️ SIH 2026 — PS 26143: U-Net Oil-Spill Training Notebook

> **Member 1 scope:** Satellite Imagery + AI Oil-Spill Detection  
> **Phase:** Model Training + Evaluation  
> **Dataset:** Sentinel-1 SAR (2-channel, 2048×2048, float32 TIFF)

---

### What this notebook does
1. Installs all dependencies
2. Mounts Google Drive / sets up the repo
3. Verifies the prototype dataset
4. Trains U-Net with BCE + Dice loss
5. Evaluates on the held-out test set
6. Runs inference on a new image

### ⚠️ Before running
- Go to **Runtime → Change runtime type → T4 GPU**
- Ensure the prototype dataset is at `/content/sih26143/prototype/`

## 0. Install Dependencies

In [ ]:
# Install all required packages
# rasterio: read Sentinel-1 GeoTIFF files
# torch / torchvision: deep learning framework
# tqdm: progress bars
!pip install -q rasterio torch torchvision tqdm matplotlib numpy

## 1. Mount Google Drive & Clone Repository

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, sys
from pathlib import Path

# ── Option A: Clone from GitHub (fill in your repo URL) ──────────────────────
GITHUB_URL = ""  # e.g. "https://github.com/your-org/sih26143-oil-spill.git"
REPO_DIR   = Path("/content/sih26143-oil-spill")

if GITHUB_URL and not REPO_DIR.exists():
    !git clone {GITHUB_URL} {REPO_DIR}

# ── Option B: Repo already on Drive ──────────────────────────────────────────
# REPO_DIR = Path("/content/drive/MyDrive/sih26143-oil-spill")

# Add to Python path
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

print(f"Repo dir : {REPO_DIR}")
print(f"Exists   : {REPO_DIR.exists()}")

## 2. Configure Dataset Path

In [ ]:
from pathlib import Path

# ── Set this to your prototype dataset root ───────────────────────────────────
DATASET_ROOT = Path("/content/sih26143/prototype")

# Alternative locations:
# DATASET_ROOT = Path("/content/drive/MyDrive/SIH2026/prototype")

# Checkpoint output
CKPT_DIR = REPO_DIR / "ml" / "training" / "checkpoints"
CKPT_DIR.mkdir(parents=True, exist_ok=True)

# Validate structure
for split in ['train', 'val', 'test']:
    for sub in ['images', 'masks']:
        d = DATASET_ROOT / split / sub
        files = list(d.glob('*.tif')) if d.exists() else []
        status = f"{len(files)} TIFFs" if d.exists() else "MISSING"
        print(f"  {split:5s}/{sub:6s}  →  {status}")

## 3. Inspect a Sample Image/Mask Pair

In [ ]:
import warnings
import numpy as np
import matplotlib.pyplot as plt
import rasterio
from rasterio.errors import NotGeoreferencedWarning
warnings.filterwarnings('ignore', category=NotGeoreferencedWarning)

# Load one train sample to verify shapes before training
img_files  = sorted((DATASET_ROOT / 'train' / 'images').glob('*.tif'))
mask_files = sorted((DATASET_ROOT / 'train' / 'masks').glob('*.tif'))

with rasterio.open(str(img_files[0])) as src:
    img_arr = src.read()   # (C, H, W)
with rasterio.open(str(mask_files[0])) as src:
    msk_arr = src.read()   # (1, H, W)

print(f"Image  : {img_files[0].name}")
print(f"  shape={img_arr.shape}  dtype={img_arr.dtype}")
print(f"  ch0  min={img_arr[0].min():.2f}  max={img_arr[0].max():.2f}")
print(f"  ch1  min={img_arr[1].min():.2f}  max={img_arr[1].max():.2f}")
print(f"\nMask   : {mask_files[0].name}")
print(f"  shape={msk_arr.shape}  dtype={msk_arr.dtype}")
print(f"  unique values: {np.unique(msk_arr).tolist()}")

# Quick visual check
from ml.training.dataset import normalise_sar_channel
ch0_n = normalise_sar_channel(img_arr[0])
ch1_n = normalise_sar_channel(img_arr[1])

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(ch0_n, cmap='gray'); axes[0].set_title('SAR Ch-0 (VV, normalised)'); axes[0].axis('off')
axes[1].imshow(ch1_n, cmap='gray'); axes[1].set_title('SAR Ch-1 (VH, normalised)'); axes[1].axis('off')
axes[2].imshow(msk_arr[0], cmap='binary'); axes[2].set_title(f'Mask  (unique={np.unique(msk_arr).tolist()})'); axes[2].axis('off')
plt.suptitle('Pre-training sanity check', fontweight='bold')
plt.tight_layout(); plt.show()

## 4. Train the U-Net

In [ ]:
# ── Training configuration ────────────────────────────────────────────────────
# Adjust batch_size based on your GPU VRAM:
#   T4  (16 GB) → batch_size=4  with base_features=64  is safe
#   P100 (16 GB) → same
#   A100 (40 GB) → batch_size=8 with base_features=64
#   CPU only     → set batch_size=1, base_features=16, epochs=5 (slow)

TRAIN_CONFIG = dict(
    data_dir      = str(DATASET_ROOT),
    ckpt_dir      = str(CKPT_DIR),
    epochs        = 50,
    batch_size    = 4,
    lr            = 1e-4,
    image_size    = 512,
    base_features = 64,
    patience      = 10,
    seed          = 42,
)

print("Training configuration:")
for k, v in TRAIN_CONFIG.items():
    print(f"  {k:15s}: {v}")

In [ ]:
import torch
print(f"PyTorch version : {torch.__version__}")
print(f"CUDA available  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU             : {torch.cuda.get_device_name(0)}")
    print(f"VRAM            : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

# Import and run the training function programmatically
# (avoids subprocess overhead and keeps notebook output inline)
import argparse
from ml.training.train import train

# Build a namespace matching train.py's argparse expectations
args = argparse.Namespace(**TRAIN_CONFIG)
history = train(args)

## 5. Plot Training History

In [ ]:
import matplotlib.pyplot as plt

if history:
    epochs_ran = [h['epoch'] for h in history]

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    # Loss
    axes[0].plot(epochs_ran, [h['train_loss'] for h in history], label='Train Loss')
    axes[0].plot(epochs_ran, [h['val_loss']   for h in history], label='Val Loss')
    axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
    axes[0].set_title('Combined Loss (BCE + Dice)')
    axes[0].legend(); axes[0].grid(alpha=0.3)

    # Dice
    axes[1].plot(epochs_ran, [h['train_dice'] for h in history], label='Train Dice')
    axes[1].plot(epochs_ran, [h['val_dice']   for h in history], label='Val Dice')
    axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Dice Score')
    axes[1].set_title('Dice Score')
    axes[1].legend(); axes[1].grid(alpha=0.3)

    # IoU
    axes[2].plot(epochs_ran, [h['train_iou'] for h in history], label='Train IoU')
    axes[2].plot(epochs_ran, [h['val_iou']   for h in history], label='Val IoU')
    axes[2].set_xlabel('Epoch'); axes[2].set_ylabel('IoU')
    axes[2].set_title('Intersection over Union')
    axes[2].legend(); axes[2].grid(alpha=0.3)

    plt.suptitle('Training History — SAR Oil-Spill U-Net', fontsize=13, fontweight='bold')
    plt.tight_layout(); plt.show()
else:
    print('No history to plot.')

## 6. Evaluate on Test Set

In [ ]:
import torch
from pathlib import Path
from ml.training.dataset  import SAROilSpillDataset
from ml.training.evaluate import load_model, evaluate_test_set, visualise_predictions

device   = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
ckpt_path = CKPT_DIR / 'best_unet.pth'

# Load test dataset
test_ds = SAROilSpillDataset(
    image_dir  = DATASET_ROOT / 'test' / 'images',
    mask_dir   = DATASET_ROOT / 'test' / 'masks',
    image_size = TRAIN_CONFIG['image_size'],
    augment    = False,
)

# Load model checkpoint
model, ckpt = load_model(ckpt_path, device)

# Quantitative evaluation
results = evaluate_test_set(model, test_ds, device, threshold=0.5)

print(f"\n{'='*40}")
print(f"  TEST SET RESULTS")
print(f"{'='*40}")
print(f"  Mean Dice : {results['mean_dice']:.4f} ± {results['std_dice']:.4f}")
print(f"  Mean IoU  : {results['mean_iou']:.4f} ± {results['std_iou']:.4f}")
print(f"{'='*40}")

## 7. Visualise Test Predictions

In [ ]:
# Display up to 5 test predictions
# Each panel: SAR Ch-0 | SAR Ch-1 | GT Mask | Pred Mask | Overlay
visualise_predictions(model, test_ds, device, n=5, threshold=0.5)

## 8. Inference on a New Image

In [ ]:
from ml.training.inference import OilSpillPredictor

# Initialise the predictor with the best checkpoint
predictor = OilSpillPredictor(
    ckpt_path  = str(ckpt_path),
    image_size = TRAIN_CONFIG['image_size'],
    threshold  = 0.5,
)

# ── Use a test image as a demo (replace with any new TIFF path) ───────────────
demo_tiff = sorted((DATASET_ROOT / 'test' / 'images').glob('*.tif'))[0]
print(f"Running inference on: {demo_tiff.name}")

binary_mask, prob_map = predictor.predict(demo_tiff)

print(f"\nResults:")
print(f"  Binary mask shape  : {binary_mask.shape}")
print(f"  Binary mask dtype  : {binary_mask.dtype}")
print(f"  Unique values      : {set(binary_mask.flatten().tolist())}")
print(f"  Probability map    : shape={prob_map.shape}  min={prob_map.min():.3f}  max={prob_map.max():.3f}")
print(f"  Oil coverage       : {100 * binary_mask.sum() / binary_mask.size:.2f}%")

In [ ]:
# Full 4-panel visualisation for the demo image
predictor.visualise(demo_tiff)

## 9. Save Results to Drive

In [ ]:
import json, shutil

# ── Save summary JSON ─────────────────────────────────────────────────────────
summary = {
    'mean_dice': results['mean_dice'],
    'mean_iou':  results['mean_iou'],
    'std_dice':  results['std_dice'],
    'std_iou':   results['std_iou'],
    'n_test':    results['n_images'],
    'threshold': results['threshold'],
    'per_image': results['per_image'],
    'train_config': TRAIN_CONFIG,
}

summary_path = CKPT_DIR / 'test_results.json'
with open(summary_path, 'w') as f:
    json.dump(summary, f, indent=2)
print(f'Test results saved to: {summary_path}')

# ── Optionally copy checkpoint to Drive ───────────────────────────────────────
DRIVE_SAVE = False  # Set to True to copy to Drive
DRIVE_DEST = Path('/content/drive/MyDrive/SIH2026/checkpoints/')

if DRIVE_SAVE:
    DRIVE_DEST.mkdir(parents=True, exist_ok=True)
    shutil.copy2(ckpt_path, DRIVE_DEST / 'best_unet.pth')
    shutil.copy2(summary_path, DRIVE_DEST / 'test_results.json')
    print(f'Checkpoint and results copied to {DRIVE_DEST}')

---

## ✅ Training Pipeline Complete

| Step | Status |
|---|---|
| Dataset loaded | ✓ |
| U-Net trained | ✓ |
| Best model saved | ✓ `checkpoints/best_unet.pth` |
| Test Dice / IoU reported | ✓ |
| Predictions visualised | ✓ |
| Inference module ready | ✓ `ml.training.inference.OilSpillPredictor` |

**Next steps:**
- Increase training data (download Parts II + III)
- Try `base_features=32` + `batch_size=8` for faster iteration
- Add test-time augmentation (TTA) in `evaluate.py`
- Integrate with AIS data (separate team member scope)

---
*SIH 2026 — Problem Statement 26143 | Member 1: Satellite Imagery + AI Detection*